### Parse Surveys

In [1]:
import ast
post_surveys = []
pre_surveys = []
users = set()
with open("combined_data/survey_log_full.txt", "r") as infile:
    for line in infile:
        user, survey = line.split("||")
        user = user.split(":")[1].strip()
        users.add(user)
        survey = survey[13:].strip()
        survey = ast.literal_eval(survey)
        survey["user"] = user
        if "POST-SURVEY" in line:
            post_surveys.append(survey)
        else:
            pre_surveys.append(survey)   

### Get Real Objective Metrics From Filtered Transcript

In [5]:
chat_stats = {}
policy = None
dialogs = {}
DIALOG_TYPES = ['cts', 'hdc', 'faq']

user_dialog_nums = {}

with open("combined_data/transcript_full-filtered.txt", "r") as transcript:
    for line in transcript:
        if "GOAL-TYPE" in line:
            current_dialog = {}
            tmp = line.split()
            user = tmp[1].strip()
            policy = tmp[3].strip(")").strip()
            current_dialog["policy"] = policy
            current_dialog['user'] = user
            current_dialog['turns'] = []
            # goal_type = tmp[-1].strip(")").strip()
        elif "USER:" in line and not "POST-NLU" in line:
            current_dialog["turns"].append(line)
        elif "SYSTEM" in line:
            current_dialog['turns'].append(line)
        elif "DIALOG END:" in line:
            current_dialog["end_condition"] = line.split(":")[1].strip()
        elif "SUBJECTIVE LENGTH" in line:
            current_dialog["sub_length"] = line.split(":")[1].strip()
        elif "SUBJECTIVE QUALITY" in line:
            current_dialog["sub_quality"] = line.split(":")[1].strip()
        elif line.strip() == "":
            # TODO: Change this line to analyse one group at a time
            if current_dialog and current_dialog["policy"] in DIALOG_TYPES:
                obj_length = len(current_dialog["turns"])
                if not policy in chat_stats:
                    chat_stats[policy] = {"length": [], "end_condition": [], "sub_length": [], "sub_quality": []}
                chat_stats[policy]["length"].append(obj_length)
                chat_stats[policy]["end_condition"].append(current_dialog["end_condition"])
                chat_stats[policy]["sub_length"].append(int(current_dialog["sub_length"]))
                chat_stats[policy]["sub_quality"].append(int(current_dialog["sub_quality"]))
                current_dialog["length"] = obj_length
                if current_dialog['user'] not in user_dialog_nums:
                    user_dialog_nums[current_dialog['user']] = 0
                user_dialog_nums[current_dialog['user']] += 1
                if not policy in dialogs:
                    dialogs[policy] = []
                dialogs[policy].append(current_dialog)
                current_dialog = {}
print(chat_stats)

{'cts': {'length': [19, 2, 8, 3, 4, 5, 2, 6, 4, 6, 8, 4, 2, 6, 2, 6, 2, 2, 5, 5, 2, 13, 8, 2, 3, 6, 2, 6, 22, 23, 21, 4, 6, 4, 4, 2, 4, 7, 4, 6, 6, 6, 13, 5, 2, 3, 4, 2, 3, 6, 2, 3, 45, 24, 25, 20, 2, 17, 4, 2, 6], 'end_condition': ['SUCCESS', 'SUCCESS', 'FAILURE', 'FAILURE', 'SUCCESS', 'SUCCESS', 'SUCCESS', 'SUCCESS', 'SUCCESS', 'SUCCESS', 'FAILURE', 'SUCCESS', 'SUCCESS', 'SUCCESS', 'SUCCESS', 'SUCCESS', 'SUCCESS', 'SUCCESS - OTHER QUESTION', 'FAILURE', 'SUCCESS', 'SUCCESS', 'FAILURE', 'SUCCESS', 'SUCCESS', 'SUCCESS', 'SUCCESS', 'SUCCESS', 'FAILURE', 'SUCCESS', 'SUCCESS', 'SUCCESS', 'SUCCESS', 'SUCCESS', 'SUCCESS', 'SUCCESS', 'FAILURE', 'SUCCESS', 'SUCCESS', 'SUCCESS', 'SUCCESS', 'FAILURE - OTHER QUESTION', 'SUCCESS', 'FAILURE', 'SUCCESS', 'SUCCESS - OTHER QUESTION', 'SUCCESS', 'SUCCESS', 'SUCCESS', 'SUCCESS', 'SUCCESS', 'SUCCESS', 'FAILURE', 'FAILURE', 'FAILURE', 'FAILURE', 'SUCCESS', 'SUCCESS', 'FAILURE', 'SUCCESS', 'SUCCESS', 'SUCCESS'], 'sub_length': [3, 3, 1, 2, 3, 3, 2, 3, 3, 3,

### Collect corpus statistics per/policy
* Number 
* Length
* #Success
* Avg. first utterance length
* Avg. all user utterance lengths

In [14]:

for policy in chat_stats:
    num_dialogs = 0
    success = 0
    avg_len_dialog = 0
    avg_len_first_utterance = 0
    avg_len_all_utterances = 0
    avg_sub_quality = 0
    avg_sub_len = 0
    
    success = chat_stats[policy]["end_condition"].count("SUCCESS") + chat_stats[policy]["end_condition"].count("SUCCESS - OTHER QUESTION")
    avg_len_dialog = sum(chat_stats[policy]["length"])
    avg_sub_quality = sum(chat_stats[policy]["sub_quality"])
    avg_sub_len = sum(chat_stats[policy]["sub_length"])

    for record in dialogs[policy]:
        num_dialogs += 1
        d = record['turns']
        user_turns = [t for t in d if "USER" in t]
        len_user_turns = [len(t[6:].split()) for t in user_turns]
        avg_len_first_utterance += len_user_turns[0]
        avg_len_all_utterances += sum(len_user_turns)/len(len_user_turns)

    avg_len_dialog = avg_len_dialog/num_dialogs
    avg_sub_len = avg_sub_len/num_dialogs
    avg_sub_quality = avg_sub_quality/num_dialogs

    avg_len_first_utterance = avg_len_first_utterance/num_dialogs
    avg_len_all_utterances = avg_len_all_utterances/num_dialogs

    print(policy)
    print(f"NUM DIALOGS: {num_dialogs}")
    print(f"SUCCESS: {success/num_dialogs*100}")
    print(f"SUBJECTIVE QUALITY: {avg_sub_quality}")
    print(f"AVG NUM TURNS: {avg_len_dialog}")
    print(f"SUBJECTIVE LENGTH: {avg_sub_len}")

    print(f"AVG LEN INITIAL UTTERANCE: {avg_len_first_utterance}")
    print(f"AVG LEN ALL UTTERANCES: {avg_len_all_utterances}")

cts
NUM DIALOGS: 61
SUCCESS: 77.04918032786885
SUBJECTIVE QUALITY: 2.8688524590163933
AVG NUM TURNS: 7.377049180327869
SUBJECTIVE LENGTH: 2.918032786885246
AVG LEN INITIAL UTTERANCE: 8.721311475409836
AVG LEN ALL UTTERANCES: 6.361865729898516
hdc
NUM DIALOGS: 66
SUCCESS: 43.93939393939394
SUBJECTIVE QUALITY: 2.409090909090909
AVG NUM TURNS: 13.318181818181818
SUBJECTIVE LENGTH: 3.0757575757575757
AVG LEN INITIAL UTTERANCE: 8.227272727272727
AVG LEN ALL UTTERANCES: 5.370540780768053
faq
NUM DIALOGS: 61
SUCCESS: 57.377049180327866
SUBJECTIVE QUALITY: 2.6065573770491803
AVG NUM TURNS: 2.262295081967213
SUBJECTIVE LENGTH: 2.278688524590164
AVG LEN INITIAL UTTERANCE: 10.163934426229508
AVG LEN ALL UTTERANCES: 10.248633879781423


In [8]:
l = []
ec = []
sl = []
sq = []

for policy in chat_stats:
    lengths = chat_stats[policy]["length"]
    end_conditions = chat_stats[policy]["end_condition"]
    sub_lengths = chat_stats[policy]["sub_length"]
    sub_qualities = chat_stats[policy]["sub_quality"]

    avg_length = sum(lengths)/len(lengths)
    avg_success = end_conditions.count("SUCCESS")/len(end_conditions)
    avg_sub_length = sum(sub_lengths)/len(sub_lengths)
    avg_sub_quality = sum(sub_qualities)/len(sub_qualities)

    l += lengths
    ec += end_conditions
    sl += sub_lengths
    sq += sub_qualities

    print(f"{policy} -- AVG. LEN: {avg_length}  AVG. SUCCESS: {avg_success}  AVG. SUBJECTIVE LENGTH: {avg_sub_length}  AVG. SUBJECTIVE QUALITY: {avg_sub_quality}")


print(f"COMBINED -- AVG. LEN: {sum(l)/len(l)}  AVG. SUCCESS: {ec.count('SUCCESS')/len(ec)}  AVG. SUBJECTIVE LENGTH: {sum(sl)/len(sl)}  AVG. SUBJECTIVE QUALITY: {sum(sq)/len(sq)}")

OPEN -- AVG. LEN: 9.564516129032258  AVG. SUCCESS: 0.7741935483870968  AVG. SUBJECTIVE LENGTH: 2.838709677419355  AVG. SUBJECTIVE QUALITY: 2.774193548387097
EASY -- AVG. LEN: 5.921875  AVG. SUCCESS: 0.625  AVG. SUBJECTIVE LENGTH: 2.71875  AVG. SUBJECTIVE QUALITY: 2.703125
HARD -- AVG. LEN: 7.983870967741935  AVG. SUCCESS: 0.3387096774193548  AVG. SUBJECTIVE LENGTH: 2.7419354838709675  AVG. SUBJECTIVE QUALITY: 2.3870967741935485
COMBINED -- AVG. LEN: 7.803191489361702  AVG. SUCCESS: 0.5797872340425532  AVG. SUBJECTIVE LENGTH: 2.765957446808511  AVG. SUBJECTIVE QUALITY: 2.622340425531915


### CTS Chat statistics

| Goal Type | Avg. Dialog Length| Avg. Success | Avg. Subjective Length | Avg. Subjective Quality |
|-----------|-------------------|--------------|------------------------|-------------------------|
| OPEN      | 9.30              | 0.90         | 3.05                   | 2.80                    |
| EASY      | 4.76              | 0.76         | 2.67                   | 2.90                    |
| HARD      | 8.32              | 0.53         | 3.00                   | 2.95                    |
| COMBINED  | 7.40              | 0.73         | 2.90                   | 2.89                    |

### FAQ Chat statistics

| Goal Type | Avg. Dialog Length| Avg. Success | Avg. Subjective Length | Avg. Subjective Quality |
|-----------|-------------------|--------------|------------------------|-------------------------|
| OPEN      | 2.20              | 0.65         | 2.20                   | 2.60                    |
| EASY      | 2.19              | 0.81         | 2.29                   | 2.71                    |
| HARD      | 2.38              | 0.24         | 2.38                   | 2.48                    |
| COMBINED  | 2.26              | 0.56         | 2.90                   | 2.60                    |

### HDC Chat statistics

| Goal Type | Avg. Dialog Length| Avg. Success | Avg. Subjective Length | Avg. Subjective Quality |
|-----------|-------------------|--------------|------------------------|-------------------------|
| OPEN      | 16.50             | 0.77         | 3.23                   | 2.91                    |
| EASY      | 10.60             | 0.32         | 3.18                   | 2.50                    |
| HARD      | 12.86             | 0.23         | 2.82                   | 1.82                    |
| COMBINED  | 13.32             | 0.44         | 3.08                   | 2.41                    |

### Remove user's who didn't interact with system

In [9]:
user_black_list = set()
user_gray_list = set()

for user in users:
    if user in user_dialog_nums:
        if user_dialog_nums[user] != 3:
            user_gray_list.add(user)
    else:
        user_black_list.add(user)

print(f"Removed {len(user_black_list)} users: {user_black_list}")
print("To Investigate: ", user_gray_list)
users = [user for user in users if user not in user_black_list]
print(len(users))


Removed 0 users: set()
To Investigate:  {'2e4bd44563cdfd591a0a4bd566c308', 'e5de0b4b0679651ef175fceeb3413f'}
21


### Save data to csv format

In [259]:
import csv
with open("pre_survey_cts.csv", "w", newline='') as outfile:
    fieldnames = pre_surveys[0].keys()
    writer = csv.DictWriter(outfile, fieldnames=fieldnames, delimiter="|")
    writer.writeheader()
    for s in pre_surveys:
        if s['user'] in user_black_list:
            continue
        writer.writerow(s)

with open("post_surveys_cts.csv", "w", newline='') as outfile:
    fieldnames = post_surveys[0].keys()
    writer = csv.DictWriter(outfile, fieldnames=fieldnames)
    writer.writeheader()
    for s in post_surveys:
        if s['user'] in user_black_list:
            continue
        writer.writerow(s)

### Parse out Trust and Usability scores

In [10]:
import numpy as np

trust = []
reliability = []
usability = []

u_usability = {}
u_trust = {}
u_reliability = {}

for res in post_surveys:
    user = res["user"]
    if user in user_black_list or user == "47e68725c26f72d805709141e76fd0" or user == "57affbcf1a53cf8152a4f84b337572":
        continue
    user_reliability = (int(res["reliability_1"]) + int(res["reliability_2"]) + (6 - int(res["reliability_3"])) + int(res["reliability_4"]) + (6 - int(res["reliability_5"])) + int(res["reliability_6"])) / 6
    user_trust = (int(res["trust_1"]) + int(res["trust_2"])) / 2
    # should be 0 to 4 scale, not 1 to 5
    user_usability = ((int(res['umux_1']) - 1) + (5 - int(res['umux_2'])) + (int(res['umux_3']) - 1) + (5 - int(res['umux_4']))) / 16 * 100
    u_usability[user] = user_usability
    trust.append(user_trust)
    u_trust[user] = user_trust
    reliability.append(user_reliability)
    u_reliability[user] = user_reliability
    usability.append(user_usability)
    
    # print(f" USER: {user}: Trust: {user_trust} Reliability: {user_reliability} Usability: {user_usability}")
    
print(f"TRUST: {np.mean(trust)} +/- {np.std(trust)}")
print(f"RELIABILITY: {np.mean(reliability)} +/- {np.std(reliability)}")
print(f"USABILITY: {np.mean(usability)} +/- {np.std(usability)}")

TRUST: 2.8253968253968256 +/- 0.9886608313041441
RELIABILITY: 2.7037037037037033 +/- 0.797064966761485
USABILITY: 51.388888888888886 +/- 26.315687946522804


### What mental models did users have?

In [11]:
import plotly.graph_objects as go
from scipy import stats

mental_models = {
      "natural language": [],
      "keywords": [],
      "specific question": [],
      "follow-up questions": [],
      "general answer": [],
      "personalized answer": [],
      "immediate answer": [],
      "long dialog": []
}

user_mms = {}


for res in pre_surveys:
    user = res["user"]
    if user not in user_black_list:
        mental_models["natural language"].append(int(res["chat_exp_1"]))
        mental_models["keywords"].append(int(res["chat_exp_2"]))
        mental_models["specific question"].append(int(res["chat_exp_3"]))
        mental_models["follow-up questions"].append(int(res['chat_exp_4']))
        mental_models["general answer"].append(int(res["chat_exp_5"]))
        mental_models["personalized answer"].append(int(res["chat_exp_6"]))
        mental_models["immediate answer"].append(int(res['chat_exp_7']))
        mental_models["long dialog"].append(int(res["chat_exp_8"]))
        user_mms[user] = {key: mental_models[key][-1] for key in mental_models}
    else:
        print(user)
    
labels = [key for key in mental_models]
avg_mental_models = [np.mean(mental_models[l]) for l in labels]
yes_count = []
no_count = []
for l in labels:
    ys = [entry for entry in mental_models[l] if entry >= 3]
    ns = [entry for entry in mental_models[l] if entry < 3]
    yes_count.append(len(ys))
    no_count.append(len(ns))
print(mental_models)
print(avg_mental_models)

fig = go.Figure()
fig.add_trace(go.Bar(
    name='Expect',
    x=labels,
    y=yes_count
))
fig.add_trace(go.Bar(
    name="Do Not Expect",
    x=labels,
    y=no_count
))
# fig.add_trace(go.Bar(
#     name='Control',
#     x=labels, y=avg_mental_models,
#     error_y=dict(type='data', array=[stats.sem(mental_models[l]) for l in labels])
# ))
fig.update_layout(
    barmode='group', 
    width=600,
    yaxis_title="# Users",
    plot_bgcolor = 'rgba(0,0,0,0)',
    legend={"orientation": "h", "yanchor": "top", "y":1.12, "xanchor":"center", "x":0.5})
fig.update_xaxes(showline=True, linewidth=2, linecolor='darkgrey', gridcolor='rgba(0,0,0,0)')
fig.update_yaxes(showline=False, linewidth=2, linecolor='darkgrey', gridcolor='darkgrey')
fig.show()

{'natural language': [4, 2, 2, 3, 2, 2, 5, 3, 4, 2, 4, 3, 5, 4, 4, 4, 2, 3, 3, 5, 4, 3, 4, 4, 1, 5, 2, 3, 4, 4, 5, 4, 4, 3, 3, 2, 5, 2, 4, 4, 1, 4, 1, 2, 3, 5, 4, 4, 5, 3, 4, 2, 4, 2, 2, 3, 3, 2, 3, 5, 5, 4, 2, 2, 2, 4], 'keywords': [3, 4, 4, 4, 4, 4, 1, 4, 2, 4, 2, 4, 2, 4, 2, 4, 4, 4, 5, 2, 4, 4, 5, 4, 4, 5, 4, 4, 2, 3, 2, 3, 5, 4, 2, 4, 2, 4, 3, 2, 4, 2, 5, 5, 4, 5, 4, 3, 2, 5, 3, 4, 4, 4, 4, 5, 4, 2, 4, 5, 5, 3, 4, 5, 5, 1], 'specific question': [4, 4, 4, 4, 5, 5, 4, 4, 5, 4, 5, 4, 4, 5, 4, 4, 5, 4, 4, 2, 4, 4, 4, 4, 4, 5, 4, 4, 2, 2, 2, 4, 5, 2, 4, 5, 2, 5, 4, 3, 5, 4, 5, 5, 4, 1, 5, 5, 4, 4, 4, 4, 3, 2, 3, 4, 4, 5, 2, 4, 4, 3, 5, 5, 4, 3], 'follow-up questions': [5, 2, 4, 4, 2, 4, 2, 4, 3, 4, 2, 4, 5, 2, 4, 4, 3, 4, 3, 3, 4, 4, 5, 4, 4, 5, 3, 4, 5, 5, 5, 3, 3, 4, 3, 3, 5, 4, 2, 4, 5, 5, 4, 4, 4, 5, 3, 5, 5, 3, 4, 4, 3, 4, 4, 2, 5, 3, 1, 5, 4, 4, 5, 4, 4, 4], 'general answer': [4, 5, 1, 3, 4, 4, 4, 5, 3, 2, 2, 4, 5, 4, 2, 3, 4, 3, 4, 2, 2, 4, 4, 4, 5, 2, 4, 2, 2, 4, 3, 4, 2, 2, 3,

### What role does Mental model have on dialog length?

In [13]:
for key in mental_models:
    yes = []
    no = []
    for d in dialogs:
        user = d["user"]
        if user not in user_black_list:
            length = d["length"]
            if user_mms[user][key] >= 3:
                yes.append(length)
            else:
                no.append(length)
    res = stats.ttest_ind(yes, no)
    print(f" {key}: {res}")
    print(np.mean(yes), np.mean(no))

 natural language: Ttest_indResult(statistic=-0.003799350408940613, pvalue=0.9969726360537272)
7.801526717557252 7.807017543859649
 keywords: Ttest_indResult(statistic=0.40377752964281066, pvalue=0.6868401573604113)
7.951388888888889 7.318181818181818
 specific question: Ttest_indResult(statistic=-0.8766221719623661, pvalue=0.3818226666255079)
7.565217391304348 9.222222222222221
 follow-up questions: Ttest_indResult(statistic=1.1643347008204419, pvalue=0.24577965442882344)
8.090909090909092 5.739130434782608
 general answer: Ttest_indResult(statistic=-0.9762132578392756, pvalue=0.3302267313732965)
7.344 8.714285714285714
 personalized answer: Ttest_indResult(statistic=-1.2078548485255731, pvalue=0.22863664659856814)
6.753623188405797 8.411764705882353
 immediate answer: Ttest_indResult(statistic=-1.2473278695509942, pvalue=0.21384553197900746)
7.42948717948718 9.625
 long dialog: Ttest_indResult(statistic=-2.737163596906908, pvalue=0.006798476433277465)
6.980645161290322 11.66666666666

### What role do expectations have on success?

In [91]:
for key in mental_models:
    yes = []
    no = []
    for d in dialogs:
        user = d["user"]
        if user in user_black_list:
            continue
        success = 1 if d["end_condition"] == "SUCCESS" else 0
        if user_mms[user][key] >= 3:
            yes.append(success)
        else:
            no.append(success)
    res = stats.ttest_ind(yes, no)
    print(f" {key}: {res}")

 natural language: Ttest_indResult(statistic=-1.2615625894698697, pvalue=0.2116850113328921)
 keywords: Ttest_indResult(statistic=0.6817786549807436, pvalue=0.4978388353437795)
 specific question: Ttest_indResult(statistic=0.8100219620698343, pvalue=0.4209292172076745)
 follow-up questions: Ttest_indResult(statistic=2.1792641132530566, pvalue=0.03299950024309895)
 general answer: Ttest_indResult(statistic=-0.5999062719669357, pvalue=0.5506869025902197)
 personalized answer: Ttest_indResult(statistic=-1.0559774152551655, pvalue=0.29494847330721174)
 immediate answer: Ttest_indResult(statistic=1.5375601245080888, pvalue=0.12908585919722063)
 long dialog: Ttest_indResult(statistic=-1.104090331129679, pvalue=0.2736878615942981)


Mental models have no impact on actual success in the cts setting

### Role of Mental Models on Usability

In [81]:
for key in mental_models:
    yes = []
    no = []
    for d in dialogs:
        user = d["user"]
        if user not in u_usability:
            continue
        usability = u_usability[user]
        if user_mms[user][key] >= 3:
            yes.append(usability)
        else:
            no.append(usability)
    res = stats.ttest_ind(yes, no)
    print(f" {key}: {res}")
    print(np.mean(yes), np.mean(no))

 natural language: Ttest_indResult(statistic=-1.4321320775297293, pvalue=0.15738289394837543)
55.96590909090909 64.70588235294117
 keywords: Ttest_indResult(statistic=-1.090277005916156, pvalue=0.28002403112957736)
56.53409090909091 63.23529411764706
 specific question: Ttest_indResult(statistic=0.3830301576588764, pvalue=0.7030739866630724)
58.92857142857143 56.25
 follow-up questions: Ttest_indResult(statistic=nan, pvalue=nan)
58.40163934426229 nan
 general answer: Ttest_indResult(statistic=-0.6474291339110548, pvalue=0.5198655421789713)
56.8014705882353 60.416666666666664
 personalized answer: Ttest_indResult(statistic=-0.39706337358767435, pvalue=0.6927534714196355)
57.03125 59.29054054054054
 immediate answer: Ttest_indResult(statistic=-2.621444821675263, pvalue=0.011120154098656935)
55.52884615384615 75.0
 long dialog: Ttest_indResult(statistic=3.5493223590857466, pvalue=0.0007654443048735936)
61.36363636363637 31.25


Natural Language/Keyword expectation and dialog length expectation had a significant effect on usability

### Role of Mental Models on Reliability

In [82]:
for key in mental_models:
    if key == "specific question":
        continue
    yes = []
    no = []
    for d in dialogs:
        user = d["user"]
        if user not in u_reliability:
            continue
        reliability = u_reliability[user]
        if user_mms[user][key] >= 3:
            yes.append(reliability)
        else:
            no.append(reliability)
    res = stats.ttest_ind(yes, no)
    print(f" {key}: {res}")
    print(np.mean(yes), np.mean(no))

 natural language: Ttest_indResult(statistic=-1.0304388908112347, pvalue=0.3070097939761558)
2.7462121212121215 2.9509803921568625
 keywords: Ttest_indResult(statistic=-0.8916583847186615, pvalue=0.37619815459541883)
2.753787878787879 2.9313725490196076
 follow-up questions: Ttest_indResult(statistic=nan, pvalue=nan)
2.803278688524589 nan
 general answer: Ttest_indResult(statistic=0.06921970297373171, pvalue=0.9450488122042161)
2.808823529411765 2.796296296296296
 personalized answer: Ttest_indResult(statistic=1.0248705257172168, pvalue=0.30960803740759535)
2.9166666666666665 2.7297297297297303
 immediate answer: Ttest_indResult(statistic=-0.13911303633432043, pvalue=0.889834594357785)
2.798076923076922 2.8333333333333335
 long dialog: Ttest_indResult(statistic=2.8175491426145993, pvalue=0.006573857524343258)
2.8818181818181814 2.083333333333333


Natural language and dialog length had a significant effect on pereceived reliability

### Effect of mental models on trust

In [83]:
for key in mental_models:
    yes = []
    no = []
    for d in dialogs:
        user = d["user"]
        if user not in u_trust:
            continue
        trust = u_trust[user]
        if user_mms[user][key] >= 3:
            yes.append(trust)
        else:
            no.append(trust)
    res = stats.ttest_ind(yes, no)
    print(f" {key}: {res}")
    print(np.mean(yes), np.mean(no))

 natural language: Ttest_indResult(statistic=0.8603481533906172, pvalue=0.3930798471930703)
2.9204545454545454 2.7058823529411766
 keywords: Ttest_indResult(statistic=-0.11987462658573536, pvalue=0.9049897340241074)
2.852272727272727 2.8823529411764706
 specific question: Ttest_indResult(statistic=1.621143018968216, pvalue=0.11031970914810925)
2.9489795918367347 2.5
 follow-up questions: Ttest_indResult(statistic=nan, pvalue=nan)
2.860655737704918 nan
 general answer: Ttest_indResult(statistic=0.6587876603499426, pvalue=0.5125950253555297)
2.926470588235294 2.7777777777777777
 personalized answer: Ttest_indResult(statistic=0.5514486068116325, pvalue=0.5834087447901244)
2.9375 2.810810810810811
 immediate answer: Ttest_indResult(statistic=0.7203955887623041, pvalue=0.4741260286321175)
2.894230769230769 2.6666666666666665
 long dialog: Ttest_indResult(statistic=2.6752061439937114, pvalue=0.009649915824434006)
2.9545454545454546 2.0


Natural langauge and expected dialog length had a significant effect on trust

In [54]:
post_mental_models = {
      "natural language": [],
      "keywords": [],
      "specific question": [],
      "follow-up questions": [],
      "general answer": [],
      "personalized answer": [],
      "immediate answer": [],
      "long dialog": []
}

post_user_mms = {}

for res in post_surveys:
    user = res["user"]
    if user in user_black_list:
            continue
    if res["chat_exp_1"] != "None":
        post_mental_models["natural language"].append(int(res["chat_exp_1"]))
    if res["chat_exp_2"] != "None":
        post_mental_models["keywords"].append(int(res["chat_exp_2"]))
    if res["chat_exp_3"] != "None":
        post_mental_models["specific question"].append(int(res["chat_exp_3"]))
    if res["chat_exp_4"] != "None":
        post_mental_models["follow-up questions"].append(int(res['chat_exp_4']))
    if res["chat_exp_5"] != "None":
        post_mental_models["general answer"].append(int(res["chat_exp_5"]))
    if res["chat_exp_6"] != "None":
        post_mental_models["personalized answer"].append(int(res["chat_exp_6"]))
    if res["chat_exp_7"] != "None":
        post_mental_models["immediate answer"].append(int(res['chat_exp_7']))
    if res["chat_exp_8"] != "None":
        post_mental_models["long dialog"].append(int(res["chat_exp_8"]))
    post_user_mms[user] = {key: post_mental_models[key][-1] for key in post_mental_models}
    
labels = [key for key in post_mental_models]
avg_post_mental_models = [np.mean(post_mental_models[l]) for l in labels]

In [55]:
for key in mental_models:
    print({f"{key}: {stats.ttest_ind(mental_models[key], post_mental_models[key])}"})

{'natural language: Ttest_indResult(statistic=0.28033098596050277, pvalue=0.7806683492514809)'}
{'keywords: Ttest_indResult(statistic=-0.9543482955111211, pvalue=0.34563950251914166)'}
{'specific question: Ttest_indResult(statistic=2.4237726026264967, pvalue=0.01997534226520323)'}
{'follow-up questions: Ttest_indResult(statistic=-1.294369603381147, pvalue=0.20296075493839177)'}
{'general answer: Ttest_indResult(statistic=-0.6412364700532214, pvalue=0.5250264189676689)'}
{'personalized answer: Ttest_indResult(statistic=-1.4509525002200236, pvalue=0.15459078143343824)'}
{'immediate answer: Ttest_indResult(statistic=0.4965635331614206, pvalue=0.6222149306594651)'}
{'long dialog: Ttest_indResult(statistic=-1.3286579139913106, pvalue=0.1916830191682154)'}


People thought that they needed to ask more general questions after interacting with the chatbot, but otherwise there were no significant changes in mental models